In [342]:
# !pip install XlsxWriter
import xlsxwriter

## Rates table

In [343]:
# Step 1: Filter and Clean Invoice Data
import pandas as pd


# 🔧 Configure which sites to process
#selected_sites = ["DIT", "SPN", "SPCP","SPW","SPT","SPHU","SPTM","PVF","SPJ","CCS","SPB","SPL","SPLV","CCSG","SPCB","SPWV","FSU","SPK","SPLA","SPD","SPTG","KFC"]  # Example: update these as needed
selected_sites =['SPW','SPT','SPJ','SPN']
focus_columns = [
    "invoice_id", "site",'invoice_commodity_quantity', "invoice_commodity_group", "invoice_commodity_description",
    "location", "model", "unit", "rate_unit", "freight_class", "applied_rate",
    "shipment_type", "realistic_optimal_method", "xgs_rate", "historical_rate"
]

# Load the invoice input data
invoice_path = "invoice_input_data_all.xlsx"  # Update path if needed
invoice_df = pd.read_excel(invoice_path)
invoice_df = invoice_df[invoice_df['rate_ratio_normal_outlier'] == 'OK' ]
print(invoice_df.shape)

# Only modelled for Georgia source of Georgia mill rates
invoice_GA_df = invoice_df[invoice_df['model'] == True]
invoice_sample_df = invoice_df[invoice_df["rate_ratio_normal_outlier"]!= 'MISSING']


invoice_GA_df = invoice_GA_df[focus_columns]
invoice_sample_df = invoice_sample_df[focus_columns]


invoice_GA_df["invoice_commodity_description"] = invoice_GA_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

invoice_sample_df["invoice_commodity_description"] = invoice_sample_df["invoice_commodity_description"].apply(
    lambda x: x.title() if str(x).strip().lower() == "carpet tiles" else x
)

# Filter input invoices to selected sites
invoice_GA_df = invoice_GA_df[invoice_GA_df["site"].isin(selected_sites)]
invoice_sample_df = invoice_sample_df[invoice_sample_df["site"].isin(selected_sites)]



(11999, 61)


In [344]:
invoice_df.head()

,invoice_id,site,new_commodity_group,invoice_commodity_quantity,multiple_commodities,priority_multiple_commodities,freight_per_invoice,invoice_total,invoice_commodity_group,invoice_commodity_description,...,realistic_optimal_cost,realistic_optimal_method,xgs_rate,historical_rate,rate_ratio_normal,pct_difference,savings_flag,freight_ratio_normal_site,rate_ratio_normal_outlier,pct_difference_outlier
1,357863,SPN,1VNL,1294.68,True,False,191.13,3810.34,1VNL,VCT,...,107.30,LTL,0.082878,0.147627,0.561398,-43.860200,LOSS,1.599905,OK,OK
2,357870,SPCP,1VNL,5759.70,False,False,602.67,15078.17,1VNL,VCT,...,724.40,LTL,0.125770,0.104636,1.201985,20.198450,SAVINGS,0.756983,OK,OK
3,357875,SPTM,1CPT,331.22,False,False,524.66,11140.10,1CPT,Carpet Tiles,...,462.02,LTL,1.394904,1.584023,0.880608,-11.939161,LOSS,1.660806,OK,OK
5,359142,SPB,1VNL,812.79,False,False,210.72,2515.68,1VNL,LVT,...,183.57,LTL,0.225852,0.259255,0.871156,-12.884396,LOSS,1.009773,OK,OK
7,359153,PSLV,1VNL,63391.78,False,False,4235.00,114000.76,1VNL,LVT,...,8017.29,HYBRID,0.126472,0.066807,1.893103,89.310272,SAVINGS,3.096658,OK,OK


In [345]:
# We are calculating 3 core variables here:
# invoice_df_mills # this is for georgia_mill_rates and georgia_xgs_rates
# invoice_df_hist #this is for historical_market_rates mills and distributors

Historical market rates have to two variables historical_all_rates and historical_georgia_rates

XGS rates will have the historical_xgs_rates and 2024_xgs_rates

We will recommended a market rate recommended_market_rate


In [346]:
invoice_sample_df.columns

Index(['invoice_id', 'site', 'invoice_commodity_quantity',
       'invoice_commodity_group', 'invoice_commodity_description', 'location',
       'model', 'unit', 'rate_unit', 'freight_class', 'applied_rate',
       'shipment_type', 'realistic_optimal_method', 'xgs_rate',
       'historical_rate'],
      dtype='object')

In [347]:
from numpy import average

def summarize_invoice_data(df: pd.DataFrame, group_cols: list[str], column_prefix: str = "") -> pd.DataFrame:
    # Filter required fields
    filtered = df[
        df["freight_class"].notna() &
        df["historical_rate"].notna() &
        df["xgs_rate"].notna() &
        df["invoice_commodity_quantity"].notna()
    ][[
        "site", 

        "rate_unit", 
        "invoice_commodity_group", 
        "invoice_commodity_description",
        "freight_class", 
        "historical_rate",
        "xgs_rate",
        "invoice_commodity_quantity"
    ]].copy()

    grouped = filtered.groupby(group_cols)

    # Quantiles
    summary_avg = grouped.agg(
        historical_xgs_median = ("xgs_rate", "median"),
        historical_xgs_q3 = ("xgs_rate", lambda x: x.quantile(0.75)),
        
        historical_market_median = ("historical_rate", "median"),
        historical_market_q3 = ("historical_rate", lambda x: x.quantile(0.75)),
    )

    # Rename with prefix
    summary_avg = summary_avg.rename(columns={
        "historical_xgs_q3": f"{column_prefix}historical_xgs_q3",
        "historical_xgs_median": f"{column_prefix}historical_xgs_median",
        "historical_market_q3": f"{column_prefix}historical_market_q3",
        "historical_market_median": f"{column_prefix}historical_market_median"
    })


    # Weighted averages
    def compute_wavg(grp):
        return pd.Series({
            f"{column_prefix}historical_market_wavg": average(grp["historical_rate"], weights=grp["invoice_commodity_quantity"]),
            f"{column_prefix}historical_xgs_wavg": average(grp["xgs_rate"], weights=grp["invoice_commodity_quantity"])
        })

    summary_wavg = grouped.apply(compute_wavg)

    # Combine
    summary = pd.concat([summary_avg, summary_wavg], axis=1)
    return summary


In [348]:
summary_georgia = summarize_invoice_data(invoice_GA_df, [
    "site", "rate_unit", "invoice_commodity_group", "invoice_commodity_description", "freight_class"
],    column_prefix="georgia_"
)
summary_georgia


georgia_historical_xgs_median  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                                  
SPJ  CWT       1VNL                    LVP                           10M                                 0.084880   
                                                                     1M                                  0.161563   
                                                                     20M                                 0.084646   
                                                                     2M                                  0.129970   
                                                                     3M                                  0.129969   
...                                                                                                           ...   
SPW  SQYD      1CPT                    Carpet Tiles                  2M                                  0.805809   
                                                                     3M                                  0.519059   
                                                                     5C                                  0.833137   
                                                                     5M                                  0.407181   
                                                                     L5C                                 0.853730   

                                                                                    georgia_historical_xgs_q3  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                              
SPJ  CWT       1VNL                    LVP                           10M                             0.084880   
                                                                     1M                              0.161564   
                                                                     20M                             0.084763   
                                                                     2M                              0.129971   
                                                                     3M                              0.129970   
...                                                                                                       ...   
SPW  SQYD      1CPT                    Carpet Tiles                  2M                              0.805809   
                                                                     3M                              0.627762   
                                                                     5C                              0.833140   
                                                                     5M                              0.420097   
                                                                     L5C                             1.708670   

                                                                                    georgia_historical_market_median  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                                     
SPJ  CWT       1VNL                    LVP                           10M                                    0.077670   
                                                                     1M                                     0.145399   
                                                                     20M                                    0.066257   
                                                                     2M                                     0.141086   
                                                                     3M                                     0.110064   
...                                                                                                              ...   
SPW  SQYD      1CPT                    Carpet Tiles                  2M                                     0.993678   
            

In [349]:
summary_georgia2 = summarize_invoice_data(invoice_GA_df, [
    "site","invoice_commodity_group",
],    column_prefix="georgia_"
)
summary_georgia2 = summary_georgia2.reset_index()
summary_georgia2.head(2)

,site,invoice_commodity_group,georgia_historical_xgs_median,georgia_historical_xgs_q3,georgia_historical_market_median,georgia_historical_market_q3,georgia_historical_market_wavg,georgia_historical_xgs_wavg
0,SPJ,1CBL,0.700063,1.231899,1.110548,2.142453,1.031672,0.532134
1,SPJ,1CPT,0.824477,1.162681,1.369865,2.511261,1.374732,0.808786


In [350]:
summary_sample = summarize_invoice_data(invoice_sample_df, [
    "site", "rate_unit", "invoice_commodity_group", "invoice_commodity_description", "freight_class"
],    column_prefix="all_states_"
)
summary_sample

all_states_historical_xgs_median  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                                     
SPJ  CWT       1VNL                    LVP                           10M                                    0.084880   
                                                                     1M                                     0.161563   
                                                                     20M                                    0.084412   
                                                                     2M                                     0.129970   
                                                                     3M                                     0.129969   
...                                                                                                              ...   
SPW  SQYD      1CPT                    Carpet Tiles                  2M                                     0.805809   
                                                                     3M                                     0.519059   
                                                                     5C                                     0.833136   
                                                                     5M                                     0.407181   
                                                                     L5C                                    0.853730   

                                                                                    all_states_historical_xgs_q3  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                                 
SPJ  CWT       1VNL                    LVP                           10M                                0.084880   
                                                                     1M                                 0.161564   
                                                                     20M                                0.084646   
                                                                     2M                                 0.129971   
                                                                     3M                                 0.129970   
...                                                                                                          ...   
SPW  SQYD      1CPT                    Carpet Tiles                  2M                                 0.805809   
                                                                     3M                                 0.627762   
                                                                     5C                                 0.833139   
                                                                     5M                                 0.420097   
                                                                     L5C                                1.722259   

                                                                                    all_states_historical_market_median  \
site rate_unit invoice_commodity_group invoice_commodity_description freight_class                                        
SPJ  CWT       1VNL                    LVP                           10M                                       0.077670   
                                                                     1M                                        0.145399   
                                                                     20M                                       0.071310   
                                                                     2M                                        0.141086   
                                                                     3M                                        0.093481   
...                                                                                                                 ...   
SPW  SQYD      1CPT           

In [351]:
summary_sample.columns

Index(['all_states_historical_xgs_median', 'all_states_historical_xgs_q3',
       'all_states_historical_market_median',
       'all_states_historical_market_q3', 'all_states_historical_market_wavg',
       'all_states_historical_xgs_wavg'],
      dtype='object')

In [352]:
summary_sample2 = summarize_invoice_data(invoice_sample_df, [
    "site","invoice_commodity_group",
],    column_prefix="all_states_"
)
summary_sample2 = summary_sample2.reset_index()

summary_sample2

,site,invoice_commodity_group,all_states_historical_xgs_median,all_states_historical_xgs_q3,all_states_historical_market_median,all_states_historical_market_q3,all_states_historical_market_wavg,all_states_historical_xgs_wavg
0,SPJ,1CBL,0.485000,0.698278,0.723750,1.336381,0.698256,0.497776
1,SPJ,1CPT,0.824479,1.162681,1.379520,2.755487,1.345779,0.811994
2,SPJ,1VNL,0.208447,0.530988,0.156364,0.633236,0.102855,0.105568
3,SPN,1CBL,0.355701,0.564854,0.869074,2.061266,0.816691,0.219773
4,SPN,1CPT,0.604754,1.152530,0.847222,2.369983,0.622094,0.427163
5,SPN,1VNL,0.149316,0.428078,0.182095,0.372382,0.078508,0.052433
6,SPT,1CBL,0.713412,1.178475,1.031778,1.438800,1.215830,0.625384
7,SPT,1CPT,0.870250,1.625302,1.396206,2.367680,1.346482,0.869243
8,SPT,1VNL,0.231241,0.327708,0.171444,0.438709,0.142318,0.127762
9,SPW,1CBL,0.534366,1.174034,1.056583,1.636838,0.635219,0.386252


In [353]:
summary_sample2.columns

Index(['site', 'invoice_commodity_group', 'all_states_historical_xgs_median',
       'all_states_historical_xgs_q3', 'all_states_historical_market_median',
       'all_states_historical_market_q3', 'all_states_historical_market_wavg',
       'all_states_historical_xgs_wavg'],
      dtype='object')

In [354]:
summary_georgia2.columns

Index(['site', 'invoice_commodity_group', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_q3', 'georgia_historical_market_median',
       'georgia_historical_market_q3', 'georgia_historical_market_wavg',
       'georgia_historical_xgs_wavg'],
      dtype='object')

In [355]:
summary_combined = pd.merge(
    summary_sample2,
    summary_georgia2,
    on=["site", "invoice_commodity_group"],
    how="outer"  # or "inner" depending on whether you want all or matching keys
)

desired_column_order = [
    "site",
    "invoice_commodity_group",

    "all_states_historical_market_median",
    "all_states_historical_market_wavg",
    "all_states_historical_market_q3",

    # "all_states_historical_xgs_median",
    # "all_states_historical_xgs_wavg",
    # "all_states_historical_xgs_q3",

    "georgia_historical_market_median",
    "georgia_historical_market_wavg",
    "georgia_historical_market_q3",

    "georgia_historical_xgs_median",
    "georgia_historical_xgs_wavg",
    "georgia_historical_xgs_q3",
]


summary_combined = summary_combined[desired_column_order]

summary_combined.columns

Index(['site', 'invoice_commodity_group',
       'all_states_historical_market_median',
       'all_states_historical_market_wavg', 'all_states_historical_market_q3',
       'georgia_historical_market_median', 'georgia_historical_market_wavg',
       'georgia_historical_market_q3', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_wavg', 'georgia_historical_xgs_q3'],
      dtype='object')

In [356]:
import pandas as pd

with pd.ExcelWriter("summary_output4.xlsx", engine="xlsxwriter") as writer:
    summary_sample2.to_excel(writer, sheet_name="All Sites Summary", index=False)
    summary_georgia2.to_excel(writer, sheet_name="Georgia Mills Summary", index=False)
    summary_combined.to_excel(writer, sheet_name="Combined Summary", index=False)


In [372]:
summary_combined.head(2)

,site,invoice_commodity_group,all_states_historical_market_median,all_states_historical_market_wavg,all_states_historical_market_q3,georgia_historical_market_median,georgia_historical_market_wavg,georgia_historical_market_q3,georgia_historical_xgs_median,georgia_historical_xgs_wavg,georgia_historical_xgs_q3
0,SPJ,1CBL,0.72375,0.698256,1.336381,1.110548,1.031672,2.142453,0.700063,0.532134,1.231899
1,SPJ,1CPT,1.37952,1.345779,2.755487,1.369865,1.374732,2.511261,0.824477,0.808786,1.162681


  site invoice_commodity_group                             source  ppt_rates
0  SPJ                    1CBL  all_states_historical_market_wavg   0.698256
1  SPJ                    1CPT  all_states_historical_market_wavg   1.345779
2  SPJ                    1VNL  all_states_historical_market_wavg   0.102855
3  SPN                    1CBL  all_states_historical_market_wavg   0.816691
4  SPN                    1CPT  all_states_historical_market_wavg   0.622094


## Create block

In [358]:
# Step 4: Create block tables for each metric (no column prefixes)

index_cols = ["site", "rate_unit", "invoice_commodity_group", "invoice_commodity_description"]
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

def safe_pivot(metric_col, source_name):
    pivoted = summary.pivot(index=index_cols, columns="freight_class", values=metric_col).reset_index()
    
    # Ensure all freight class columns are present
    for fc in freight_classes:
        if fc not in pivoted.columns:
            pivoted[fc] = None

    # Add required template columns
    pivoted.rename(columns={
        "rate_unit": "unit",
        "invoice_commodity_group": "commodity_group",
        "invoice_commodity_description": "commodity_description"
    }, inplace=True)
    pivoted["site_description"] = pivoted['site']
    pivoted["unitclass"] = pivoted["unit"].apply(lambda x: "Weight" if x == "CWT" else "Area")
    pivoted["source"] = source_name

    # Reorder
    ordered_cols = ["site_description", "site", "unit", "unitclass", "commodity_group", "commodity_description"] + freight_classes + ["source"]
    return pivoted[ordered_cols]



In [359]:
summary.columns

Index(['site', 'rate_unit', 'invoice_commodity_group',
       'invoice_commodity_description', 'freight_class',
       'georgia_historical_xgs_median', 'georgia_historical_xgs_q3',
       'georgia_historical_market_median', 'georgia_historical_market_q3',
       'georgia_historical_market_wavg', 'georgia_historical_xgs_wavg',
       'all_states_historical_xgs_median', 'all_states_historical_xgs_q3',
       'all_states_historical_market_median',
       'all_states_historical_market_q3', 'all_states_historical_market_wavg',
       'all_states_historical_xgs_wavg'],
      dtype='object')

In [360]:


# Generate four blocks

# Historical Georgia Mills

# Pivot each metric using safe_pivot(metric_column_name, source_name)
# Ordered: median → wavg → q3 per group

# Georgia — historical market
georgia_market_median = safe_pivot("georgia_historical_market_median", "georgia_historical_market_median")
georgia_market_wavg = safe_pivot("georgia_historical_market_wavg", "georgia_historical_market_wavg")
georgia_market_q3 = safe_pivot("georgia_historical_market_q3", "georgia_historical_market_q3")

# Georgia — xgs
georgia_xgs_median = safe_pivot("georgia_historical_xgs_median", "georgia_historical_xgs_median")
georgia_xgs_wavg = safe_pivot("georgia_historical_xgs_wavg", "georgia_historical_xgs_wavg")
georgia_xgs_q3 = safe_pivot("georgia_historical_xgs_q3", "georgia_historical_xgs_q3")

# All states — historical market
all_market_median = safe_pivot("all_states_historical_market_median", "all_states_historical_market_median")
all_market_wavg = safe_pivot("all_states_historical_market_wavg", "all_states_historical_market_wavg")
all_market_q3 = safe_pivot("all_states_historical_market_q3", "all_states_historical_market_q3")

# All states — xgs
all_xgs_median = safe_pivot("all_states_historical_xgs_median", "all_states_historical_xgs_median")
all_xgs_wavg = safe_pivot("all_states_historical_xgs_wavg", "all_states_historical_xgs_wavg")
all_xgs_q3 = safe_pivot("all_states_historical_xgs_q3", "all_states_historical_xgs_q3")

# Combine all in desired order
combined_output = pd.concat([
    georgia_market_median,
    georgia_market_wavg,
    georgia_market_q3,

    georgia_xgs_median,
    georgia_xgs_wavg,
    georgia_xgs_q3,

    all_market_median,
    all_market_wavg,
    all_market_q3,

    all_xgs_median,
    all_xgs_wavg,
    all_xgs_q3,
], ignore_index=True)


# Sort for visual clarity
combined_output = combined_output.sort_values(by=["commodity_group", "commodity_description", "site", "unit", "source"]).reset_index(drop=True)

# Preview
print("✅ Combined pivot output (clean format):")
combined_output.head()


✅ Combined pivot output (clean format):


freight_class,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
2,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.485016,0.472000,0.465447,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_xgs_median
4,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.784225,0.472003,0.465447,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_xgs_q3


In [361]:
# Step 3: Ensure All Required Columns in Combined Output

freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Ensure all freight class columns exist in the output
for col in freight_classes:
    if col not in combined_output.columns:
        combined_output[col] = None

# Ensure proper column order
ordered_cols = [
    "site_description", "site", "unit", "unitclass", "commodity_group", "commodity_description"
] + freight_classes + ["source"]

combined_output = combined_output[ordered_cols]

# Preview the cleaned, structured result
print("✅ Final Structured Invoice Summary (Step 3):")
combined_output.head()


✅ Final Structured Invoice Summary (Step 3):


freight_class,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
1,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
2,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.485016,0.472000,0.465447,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_xgs_median
4,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.784225,0.472003,0.465447,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_xgs_q3


## Get XGS dicounted rates an cimbine

In [362]:
# Step 4: Add Source Column and Append to Vendor Data

# Load vendor data
vendor_path = "freight_rates_operating_multi_reporting_all.csv"  # Update path if needed
vendor_df = pd.read_csv(vendor_path)

# Filter vendor freight rates to selected sites
vendor_df = vendor_df[vendor_df["site"].isin(selected_sites)]


# Add source tag
vendor_df["source"] = "vendor"

# Ensure all required freight class columns exist in vendor_df
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']
for col in freight_classes:
    if col not in vendor_df.columns:
        vendor_df[col] = None

# Ensure consistent column ordering
final_cols = [
    "site_description", "site", "unit", "unitclass", "commodity_group", "commodity_description"
] + freight_classes + ["source"]

vendor_df = vendor_df[final_cols]

# Structuring column names
combined_output = combined_output[final_cols]  # Already structured in prior step

# Append invoice summary blocks to vendor table
combined_df = pd.concat([vendor_df, combined_output], ignore_index=True)

# Preview the result
print("✅ Appended Final Table (Step 4):")
combined_df.tail()


✅ Appended Final Table (Step 4):


,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
255,SPW,SPW,CWT,Weight,1VNL,VCT,6.069365,0.150482,0.209069,0.141993,0.143557,0.141505,0.092424,0.072092,NaN,NaN,georgia_historical_market_q3
256,SPW,SPW,CWT,Weight,1VNL,VCT,0.704173,0.172783,0.166306,0.115851,0.142626,0.120388,0.081298,0.061510,NaN,NaN,georgia_historical_market_wavg
257,SPW,SPW,CWT,Weight,1VNL,VCT,0.911998,0.231245,0.179358,0.141366,0.141366,0.112973,0.094178,0.078475,NaN,NaN,georgia_historical_xgs_median
258,SPW,SPW,CWT,Weight,1VNL,VCT,6.049750,0.231248,0.179359,0.141367,0.141368,0.112974,0.094178,0.085668,NaN,NaN,georgia_historical_xgs_q3
259,SPW,SPW,CWT,Weight,1VNL,VCT,0.636287,0.231245,0.179358,0.141366,0.141367,0.112973,0.094178,0.081745,NaN,NaN,georgia_historical_xgs_wavg


In [363]:
# Step 6: Normalize Vendor Rates from $/CWT to $/LBS

# Identify rows where unit is CWT (used for 1VNL)
vendor_cwt_mask = (combined_df["commodity_group"] == "1VNL") & (combined_df["unit"] == "CWT")

# List of freight class columns to scale
freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Convert vendor rates from $/CWT to $/LBS
combined_df.loc[vendor_cwt_mask, freight_class_cols] = combined_df.loc[vendor_cwt_mask, freight_class_cols] / 100

print("✅ Converted vendor CWT rates to $/LBS for comparability.")


✅ Converted vendor CWT rates to $/LBS for comparability.


In [364]:
combined_df['source'].unique()

array(['vendor', 'all_states_historical_market_median',
       'all_states_historical_market_q3',
       'all_states_historical_market_wavg',
       'all_states_historical_xgs_median', 'all_states_historical_xgs_q3',
       'all_states_historical_xgs_wavg',
       'georgia_historical_market_median', 'georgia_historical_market_q3',
       'georgia_historical_market_wavg', 'georgia_historical_xgs_median',
       'georgia_historical_xgs_q3', 'georgia_historical_xgs_wavg'],
      dtype=object)

In [365]:
combined_df['commodity_description'].unique()

array(['LVT', 'LVP', 'VCT', 'Carpet Roll', 'Carpet Tiles'], dtype=object)

## Get sample size per freight class

In [366]:
# Define freight classes in correct order
freight_classes = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# Group by site, commodity, and freight class → count unique invoice_ids
invoice_counts = invoice_df.groupby(
    ["site", "invoice_commodity_description", "freight_class"]
)["invoice_id"].nunique().reset_index(name="invoice_count")

# List to collect all site-level matrices
invoice_matrix_list = []

# Loop over sites
for site in invoice_counts["site"].unique():
    site_df = invoice_counts[invoice_counts["site"] == site]

    # Pivot per site
    matrix = site_df.pivot_table(
        index="invoice_commodity_description",
        columns="freight_class",
        values="invoice_count",
        fill_value=0
    )

    # Reindex to ensure all freight classes are present
    matrix = matrix.reindex(columns=freight_classes, fill_value=0)
    matrix = matrix.reset_index()

    # Add required columns
    matrix["site_description"] = "Itasca"  # Adjust dynamically if needed
    matrix["site"] = site
    matrix["unit"] = None
    matrix["unitclass"] = None
    matrix["commodity_group"] = None
    matrix["source"] = "invoice_counts"

    # Reorder to match combined_df structure
    final = matrix[[
        "site_description", "site", "unit", "unitclass",
        "commodity_group", "invoice_commodity_description"
    ] + freight_classes + ["source"]]

    final.rename(columns={"invoice_commodity_description": "commodity_description"}, inplace=True)
    invoice_matrix_list.append(final)

# Concatenate all site-level matrices
invoice_matrix_final = pd.concat(invoice_matrix_list, ignore_index=True)

# ✅ Now append to combined_df
combined_df = pd.concat([combined_df, invoice_matrix_final], ignore_index=True)

combined_df


,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,Jacksonville,SPJ,CWT,Weight,1VNL,LVT,0.2287,0.2085,0.1616,0.1300,0.1300,0.1019,0.0849,0.0849,0.0849,0.0849,vendor
1,Jacksonville,SPJ,CWT,Weight,1VNL,LVP,0.2287,0.2085,0.1616,0.1300,0.1300,0.1019,0.0849,0.0849,0.0849,0.0849,vendor
2,Jacksonville,SPJ,CWT,Weight,1VNL,VCT,0.2287,0.2085,0.1616,0.1300,0.1300,0.1019,0.0849,0.0849,0.0849,0.0849,vendor
3,Jacksonville,SPJ,SQYD,Area,1CBL,Carpet Roll,0.4366,0.4249,0.4190,0.4102,0.3955,0.3955,0.3955,0.3955,0.3955,0.3955,vendor
4,Jacksonville,SPJ,SQYD,Area,1CPT,Carpet Tiles,0.7422,0.7223,0.7123,0.6974,0.6724,0.6724,0.6724,0.6724,0.6724,0.6724,vendor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Itasca,SPWV,None,None,None,Carpet Tiles,285.0000,24.0000,10.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts
496,Itasca,SPWV,None,None,None,LVP,30.0000,9.0000,4.0000,2.0000,0.0000,1.0000,1.0000,0.0000,1.0000,0.0000,invoice_counts
497,Itasca,SPWV,None,None,None,LVT,38.0000,13.0000,12.0000,2.0000,3.0000,1.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts
498,Itasca,SPWV,None,None,None,VCT,14.0000,7.0000,1.0000,1.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts


## Rearrange the rates

In [367]:
# Final Sort: Enforce output row order for readability

# Define source display order
source_order = {
    "vendor": 1,

    # Georgia - historical market
    "georgia_historical_market_median": 2,
    "georgia_historical_market_wavg": 3,
    "georgia_historical_market_q3": 4,

    # All states - historical market
    "all_states_historical_market_median": 5,
    "all_states_historical_market_wavg": 6,
    "all_states_historical_market_q3": 7,

    # Georgia - xgs
    "georgia_historical_xgs_median": 8,
    "georgia_historical_xgs_wavg": 9,
    "georgia_historical_xgs_q3": 10,

    # All states - xgs
    "all_states_historical_xgs_median": 11,
    "all_states_historical_xgs_wavg": 12,
    "all_states_historical_xgs_q3": 13,

    "invoice_counts": 99  # Always last
}

# Add sorting key column
combined_df["source_sort"] = combined_df["source"].map(source_order)

# Sort rows to follow commodity hierarchy and defined source order
combined_df = combined_df.sort_values(
    by=["commodity_group", "commodity_description", "site", "unit", "source_sort"]
).drop(columns="source_sort")

# Reset index for cleanliness
combined_df.reset_index(drop=True, inplace=True)

print("✅ Rows sorted for visual clarity.")
display(combined_df.head(20))  # Display first 20 rows for quick check


✅ Rows sorted for visual clarity.


,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source
0,Jacksonville,SPJ,SQYD,Area,1CBL,Carpet Roll,0.436600,0.424900,0.419000,0.410200,0.3955,0.395500,0.3955,0.3955,0.3955,0.3955,vendor
1,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median
2,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.243128,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_wavg
3,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3
4,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median
5,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.819843,0.572386,0.399972,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_wavg
6,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.422501,0.617497,0.401370,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_q3
7,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.811437,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_median
8,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.700245,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_wavg
9,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.385038,0.472000,NaN,0.455669,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_xgs_q3


In [376]:
# Extract relevant subset
small_table = summary_combined[[
    "site",
    "invoice_commodity_group",
    "all_states_historical_market_wavg",
    "georgia_historical_market_wavg",
    "georgia_historical_xgs_wavg"
]].copy()

# Melt into long format
small_table_long = small_table.melt(
    id_vars=["site", "invoice_commodity_group"],
    value_vars=[
        "all_states_historical_market_wavg",
        "georgia_historical_market_wavg",
        "georgia_historical_xgs_wavg"
    ],
    var_name="source",
    value_name="ppt_rates"
)

# ✅ Rename the column
small_table_long = small_table_long.rename(columns={"invoice_commodity_group": "commodity_group"})

# Preview
print(small_table_long.head())


  site commodity_group                             source  ppt_rates
0  SPJ            1CBL  all_states_historical_market_wavg   0.698256
1  SPJ            1CPT  all_states_historical_market_wavg   1.345779
2  SPJ            1VNL  all_states_historical_market_wavg   0.102855
3  SPN            1CBL  all_states_historical_market_wavg   0.816691
4  SPN            1CPT  all_states_historical_market_wavg   0.622094


In [377]:
merged_df = pd.merge(
    combined_df,
    small_table_long,
    on=["site", "commodity_group", "source"],
    how="left"
)
merged_df

,site_description,site,unit,unitclass,commodity_group,commodity_description,L5C,5C,1M,2M,3M,5M,10M,20M,30M,40M,source,ppt_rates
0,Jacksonville,SPJ,SQYD,Area,1CBL,Carpet Roll,0.436600,0.424900,0.419000,0.410200,0.3955,0.3955,0.3955,0.3955,0.3955,0.3955,vendor,NaN
1,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.242723,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_median,NaN
2,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,1.243128,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_wavg,1.031672
3,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,2.389558,0.723750,NaN,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,georgia_historical_market_q3,NaN
4,SPJ,SPJ,SQYD,Area,1CBL,Carpet Roll,0.900000,0.607686,0.400141,1.000953,NaN,NaN,NaN,NaN,NaN,NaN,all_states_historical_market_median,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,Itasca,SPT,None,None,None,carpet tiles,39.000000,2.000000,3.000000,0.000000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts,NaN
496,Itasca,SPTG,None,None,None,carpet tiles,12.000000,0.000000,0.000000,1.000000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts,NaN
497,Itasca,SPTM,None,None,None,carpet tiles,33.000000,3.000000,1.000000,1.000000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts,NaN
498,Itasca,SPW,None,None,None,carpet tiles,14.000000,1.000000,1.000000,0.000000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,invoice_counts,NaN


## Final rates table

In [379]:
# 🔄 Save combined_df with each site as a separate Excel sheet

import pandas as pd

# Set export path
output_path = "19062025_freight_rates_by_site_nigel_v2300.xlsx"  # Change path if needed

# Get unique sites
sites = merged_df["site"].dropna().unique()

# Export to Excel with one sheet per site
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    for site in sites:
        sheet_name = str(site)[:31]  # Excel sheet names must be ≤ 31 characters
        site_df = merged_df[merged_df["site"] == site]
        site_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"✅ Exported to {output_path} with one sheet per site.")


✅ Exported to 19062025_freight_rates_by_site_nigel_v2300.xlsx with one sheet per site.


In [369]:
# # Use vendor_df to create a mapping from site to site_description
# site_desc_map = vendor_df.set_index("site")["site_description"].to_dict()

# # Update site_description in all relevant DataFrames by matching on 'site'
# for df_name in ["hist_wavg_block", "xgs_avg_block", "xgs_wavg_block", "variance_data", "combined_df", "invoice_matrix_final"]:
#     df = globals()[df_name]
#     df["site_description"] = df["site"].map(site_desc_map).fillna(df["site_description"])
# display(combined_df.tail(20))

In [370]:
# Step X: Append Variance Rows Between hist_invoice and xgs_invoice

# # Columns used to match rows
# index_cols = [
#     "site_description", "site", "unit", "unitclass",
#     "commodity_group", "commodity_description"
# ]

# # Freight class columns to compute variance on
# freight_class_cols = ['L5C', '5C', '1M', '2M', '3M', '5M', '10M', '20M', '30M', '40M']

# # Separate historical and xgs rows
# hist_df = combined_df[combined_df["source"] == "hist_wavg"]
# xgs_df = combined_df[combined_df["source"] == "xgs_wavg"]

# # Merge them on the index columns
# variance_df = pd.merge(hist_df, xgs_df, on=index_cols, suffixes=("_hist", "_xgs"))

# # Compute variance
# variance_data = variance_df[index_cols].copy()
# for col in freight_class_cols:
#     variance_data[col] = variance_df[f"{col}_hist"] - variance_df[f"{col}_xgs"]

# # Add source column
# variance_data["source"] = "variance"

# # Append to combined table
# combined_df = pd.concat([combined_df, variance_data], ignore_index=True)

# # Optional: sort for clarity
# combined_df.sort_values(by=index_cols + ["source"], inplace=True)

# # Preview result
# print("✅ Variance rows added.")
# combined_df.tail()
